# Async Python

## A briefing on asynchronous python coding, essential in Agent engineering

Here is a masterful tutorial by you-know-who with exercises and comparisons.

https://chatgpt.com/share/680648b1-b0a0-8012-8449-4f90b540886c

This includes how to run async code from a python module.

### And now some examples:

In [1]:
# Let's define an async function

import asyncio

async def do_some_work():
    print("Starting work")
    await asyncio.sleep(1)
    print("Work complete")


In [2]:
# What will this do?

do_some_work()

<coroutine object do_some_work at 0x7fd4d6e57b80>

In [3]:
# OK let's try that again!

await do_some_work()

Starting work
Work complete


In [4]:
# What's wrong with this?

async def do_a_lot_of_work():
    do_some_work()
    do_some_work()
    do_some_work()

await do_a_lot_of_work()

/tmp/ipykernel_69166/1833959015.py:4: RuntimeWarning: coroutine 'do_some_work' was never awaited
  do_some_work()
/tmp/ipykernel_69166/1833959015.py:5: RuntimeWarning: coroutine 'do_some_work' was never awaited
  do_some_work()
/tmp/ipykernel_69166/1833959015.py:6: RuntimeWarning: coroutine 'do_some_work' was never awaited
  do_some_work()


In [5]:
# Interesting warning! Let's fix it

async def do_a_lot_of_work():
    await do_some_work()
    await do_some_work()
    await do_some_work()

await do_a_lot_of_work()

Starting work
Work complete
Starting work
Work complete
Starting work
Work complete


In [6]:
# And now let's do it in parallel
# It's important to recognize that this is not "multi-threading" in the way that you may be used to
# The asyncio library is running on a single thread, but it's using a loop to switch between tasks while one is waiting

async def do_a_lot_of_work_in_parallel():
    await asyncio.gather(do_some_work(), do_some_work(), do_some_work())

await do_a_lot_of_work_in_parallel()

Starting work
Starting work
Starting work
Work complete
Work complete
Work complete


### Finally - try writing a python module that calls do_a_lot_of_work_in_parallel

See the link at the top; you'll need something like this in your module:

```python
if __name__ == "__main__":
    asyncio.run(do_a_lot_of_work_in_parallel())
```

In [7]:
import asyncio

async def say_hello():
    print("Hello")
    await asyncio.sleep(1)  # Pause here, switch to other tasks
    print("World")

asyncio.run(say_hello())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [8]:
import asyncio

async def greet(name):
    print(f"Hello, {name}")
    await asyncio.sleep(1)
    print(f"Goodbye, {name}")

async def main():
    await greet("Alice")

asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [10]:
import asyncio

async def hello():
    await asyncio.sleep(1)
    print("Hello from notebook!")

await hello()  # works fine!

Hello from notebook!


In [11]:
import gradio as gr
import asyncio

async def slow_response(name):
    await asyncio.sleep(2)
    return f"Hello, {name}! (after waiting)"

gr.Interface(fn=slow_response, inputs="text", outputs="text").launch()

/home/archworker1/Project_venvs_py/ai-agent-mcp/lib/python3.12/site-packages/yaml/emitter.py:31: RuntimeWarning: coroutine 'say_hello' was never awaited
  class Emitter:
/home/archworker1/Project_venvs_py/ai-agent-mcp/lib/python3.12/site-packages/yaml/emitter.py:31: RuntimeWarning: coroutine 'main' was never awaited
  class Emitter:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [2]:
import asyncio

async def task(name, delay):
    await asyncio.sleep(delay)
    print(f"{name} done after {delay}s")
    return name

async def main():
    results = await asyncio.gather(
        task("A", 1),
        task("B", 4),
        task("C", 3)
    )
    print(results)

# asyncio.run(main())
await main()

A done after 1s
C done after 3s
B done after 4s
['A', 'B', 'C']


## 🧠 Think in terms of **requests, not sessions**

When 3–5 users hit your dashboard:

* Each user request = a **separate coroutine/task**
* Your async server (like FastAPI, aiohttp, etc.) handles them using **one event loop**

---

## 🚀 What actually happens

Let’s say:

* 5 users hit your API at the same time
* Each request does:

  * DB call
  * API call
  * maybe some file read

### With async:

* All 5 requests are **handled concurrently**
* While request 1 is waiting on DB:

  * request 2 runs
* While request 2 waits on API:

  * request 3 runs
* etc.

👉 So yes — **asyncio scales across users**, not just inside one function

---

## 🔥 Where `asyncio.gather()` fits

Two different levels:

### 1. Inside a single request (what you wrote)

```python
await asyncio.gather(api1(), api2(), api3())
```

✔️ Parallel work *within one user request*

---

### 2. Across multiple users

Handled by the **async web server**

Example with FastAPI:

```python
@app.get("/data")
async def get_data():
    result = await some_async_function()
    return result
```

If 5 users call `/data`:

* FastAPI schedules **5 coroutines**
* All run concurrently on same event loop

---

## ⚠️ When it WON’T help (important)

If your code is:

```python
def get_data():
    time.sleep(5)   # blocking
```

Then:

* ❌ Whole server blocks
* ❌ Other users wait


## 💡 Key takeaway

> Async helps at **two levels**:

1. **Within a request** → `asyncio.gather`
2. **Across requests (users)** → async web server

